# Module 2 — Formal Foundations
### Stationarity · Lag Operator · ACF & PACF · Causality & Invertibility · Wold Decomposition

> **Based on:** *Univariate Time Series Analysis*, SoSe 2024 — Sections 3 & 4  
> **Data sources:** `yfinance` (DAX, EUR/USD) · `fredapi` (FRED macro) · Bundesbank via DBnomics  
> **Goal:** Build the rigorous mathematical backbone of ARMA modeling, applied to real financial and macro data

---

## Contents
1. [Setup](#0)
2. [The Lag Operator & Difference Operator](#1)
3. [Weak Stationarity — Formal Definition](#2)
4. [AR, MA, and ARMA Processes — The Algebra](#3)
5. [Stationarity Conditions for AR(p)](#4)
6. [Causality and Invertibility](#5)
7. [The Autocovariance Function (ACVF) — Derivation](#6)
8. [The Autocorrelation Function (ACF) in Practice](#7)
9. [The Partial Autocorrelation Function (PACF)](#8)
10. [The Wold Decomposition Theorem](#9)
11. [Ergodicity — Why Sample Moments Work](#10)

---

<a id='0'></a>
## Setup

In [ ]:
# !pip install yfinance fredapi statsmodels pandas matplotlib scipy

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from numpy.polynomial import polynomial as P

import yfinance as yf
from fredapi import Fred
import statsmodels.api as sm
from statsmodels.tsa.stattools import acf, pacf, adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima_process import ArmaProcess
from statsmodels.tsa.arima.model import ARIMA

plt.rcParams.update({
    'figure.figsize': (12, 4),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11
})
TEAL  = '#1D9E75'
BLUE  = '#185FA5'
AMBER = '#BA7517'
CORAL = '#D85A30'
GRAY  = '#888780'
PINK  = '#D4537E'

FRED_KEY = 'YOUR_FRED_API_KEY_HERE'
fred = Fred(api_key=FRED_KEY)

# ── Download data once, reuse throughout ──────────────────────────────────────
dax      = yf.download('^GDAXI',  start='1990-01-02', end='2024-12-31', progress=False)
eurusd   = yf.download('EURUSD=X', start='2000-01-03', end='2024-12-31', progress=False)

dax_close  = dax['Close'].squeeze()
dax_ret    = np.log(dax_close).diff().dropna()
eurusd_cls = eurusd['Close'].squeeze().dropna()
eurusd_ret = np.log(eurusd_cls).diff().dropna()

print('Data downloaded:')
print(f'  DAX   : {len(dax_close):,} daily observations')
print(f'  EURUSD: {len(eurusd_cls):,} daily observations')

<a id='1'></a>
---
## 1 · The Lag Operator and Difference Operator

### 📖 Theory

The **lag operator** $L$ (also called the backshift operator $B$) is the algebraic tool that makes time series notation compact. It shifts the time index backward by one period:

$$L y_t = y_{t-1} \qquad L^k y_t = y_{t-k} \qquad L^{-k} y_t = y_{t+k}$$

Key properties:
- $Lc = c$ for any constant $c$
- $(L^i + L^j)y_t = L^i y_t + L^j y_t$ — distributive law
- $L^i(L^j y_t) = L^{i+j} y_t$ — associative law

This lets us write an AR(p) very compactly as:
$$y_t = \alpha_1 y_{t-1} + \ldots + \alpha_p y_{t-p} + \varepsilon_t \quad \Longleftrightarrow \quad \alpha(L)\, y_t = \varepsilon_t$$

where $\alpha(L) = 1 - \alpha_1 L - \alpha_2 L^2 - \ldots - \alpha_p L^p$ is the **AR polynomial**.

#### The Difference Operator

$$\Delta y_t = y_t - y_{t-1} = (1 - L)y_t$$
$$\Delta^2 y_t = \Delta(\Delta y_t) = y_t - 2y_{t-1} + y_{t-2} = (1-L)^2 y_t$$
$$\Delta^d y_t = (1-L)^d y_t$$

Differencing $d$ times is what turns a non-stationary I($d$) series into a stationary one. For most financial series, $d=1$ (prices → returns) suffices.

#### The Characteristic Equation

Replacing $L$ with the scalar $\lambda$ in $\alpha(L)$ gives:
$$\alpha(\lambda) = 1 - \alpha_1 \lambda - \alpha_2 \lambda^2 - \ldots - \alpha_p \lambda^p = 0$$

The **roots** of this equation determine whether the process is stationary. This is the bridge between algebra and probability.

### 📈 Trading perspective
When your trading model uses an AR(2) process, the characteristic equation tells you immediately whether the model implies mean-reversion (both roots outside unit circle) or explosive behavior (at least one root inside). A stationarity check on your model parameters is a basic sanity test before going live.

In [ ]:
# ── Demonstrate the lag and difference operators on real data ─────────────────

# Use EUR/USD daily closing price
fx = eurusd_cls['2020':'2022'].copy()

# Lag operator: L^k y_t = y_{t-k}
fx_lag1 = fx.shift(1)   # L^1 y_t = y_{t-1}
fx_lag5 = fx.shift(5)   # L^5 y_t = y_{t-5}  (approx. 1 week)

# Difference operator: Δy_t = y_t - y_{t-1} = (1-L)y_t
fx_d1 = fx.diff(1)      # Δy_t   — daily changes (≈ returns for small Δ)
fx_d2 = fx.diff(1).diff(1)  # Δ²y_t  — second differences

fig, axes = plt.subplots(2, 2, figsize=(14, 7))

axes[0,0].plot(fx, color=BLUE, lw=1)
axes[0,0].plot(fx_lag1, color=CORAL, lw=1, ls='--', alpha=0.7)
axes[0,0].set_title('EUR/USD level and $Ly_t = y_{t-1}$ (1-day lag)')
axes[0,0].legend(['$y_t$', '$Ly_t$'])

axes[0,1].plot(fx, color=BLUE, lw=1)
axes[0,1].plot(fx_lag5, color=AMBER, lw=1, ls='--', alpha=0.7)
axes[0,1].set_title('EUR/USD and $L^5 y_t = y_{t-5}$ (5-day / weekly lag)')
axes[0,1].legend(['$y_t$', '$L^5 y_t$'])

axes[1,0].plot(fx_d1, color=TEAL, lw=0.8)
axes[1,0].axhline(0, color=GRAY, lw=0.5)
axes[1,0].set_title('First difference  $\\Delta y_t = (1-L)y_t$  — daily FX changes')

axes[1,1].plot(fx_d2, color=PINK, lw=0.8)
axes[1,1].axhline(0, color=GRAY, lw=0.5)
axes[1,1].set_title('Second difference  $\\Delta^2 y_t = (1-L)^2 y_t$')

plt.tight_layout()
plt.show()

print('Key insight: EUR/USD level is non-stationary (trending).')
print('First differences are approximately stationary — this is the I(1) property.')
print('Second differences are "over-differenced" (more noise, no extra gain).')

In [ ]:
# ── Characteristic roots: hands-on computation ────────────────────────────────
# For an AR(p), the characteristic equation is:  1 - α₁λ - α₂λ² - ... = 0
# We find the roots and check if |root| > 1 (stationarity condition).

def char_roots(ar_coeffs):
    """
    Compute characteristic roots of an AR(p) process.
    ar_coeffs: list [α₁, α₂, ..., αₚ] (without the leading 1)
    Returns roots of 1 - α₁λ - α₂λ² - ...
    The stationarity condition requires |root| > 1 for ALL roots.
    """
    # numpy.roots expects coefficients in descending order of degree
    # polynomial: -αₚ λᵖ - ... - α₁ λ + 1  (multiply by -1 and reverse)
    poly = np.array([1] + [-a for a in ar_coeffs])
    poly_rev = poly[::-1]  # descending order
    roots = np.roots(poly_rev)
    return roots

processes = {
    'AR(1): α=0.5  (stationary)':       [0.5],
    'AR(1): α=0.99 (near unit root)':   [0.99],
    'AR(1): α=1.0  (unit root = RW)':   [1.0],
    'AR(1): α=1.5  (explosive)':        [1.5],
    'AR(2): α₁=0.5, α₂=0.3':           [0.5, 0.3],
    'AR(2): α₁=1.2, α₂=-0.5':          [1.2, -0.5],
}

print(f'{"Process":<40} {"Roots (modulus)":<35} {"Stationary?"}')
print('-' * 90)
for name, coeffs in processes.items():
    roots = char_roots(coeffs)
    mods  = np.abs(roots)
    stat  = 'YES ✓' if np.all(mods > 1) else 'NO  ✗'
    root_str = '  '.join([f'|λ|={m:.3f}' for m in mods])
    print(f'{name:<40} {root_str:<35} {stat}')

<a id='2'></a>
---
## 2 · Weak Stationarity — Formal Definition

### 📖 Theory

**Weak (covariance) stationarity** is the workhorse concept. A time series $\{y_t\}_{t\in\mathbb{Z}}$ is weakly stationary if and only if, for all $t$:

1. **Constant mean:** $E(y_t) = \mu \quad (|\mu| < \infty)$
2. **Constant variance:** $\text{Var}(y_t) = \gamma(0) < \infty$
3. **Covariance depends only on the lag $h$, not on time $t$:** $\text{Cov}(y_t, y_{t-h}) = \gamma(h)$

Compare with **strict stationarity**: the entire joint distribution of $(y_{t_1}, \ldots, y_{t_k})$ is invariant to time shifts. Strict stationarity + finite second moments → weak stationarity (but not vice versa).

**Why it matters:** All standard ARMA theory — ACF computation, MLE, Yule-Walker equations, forecasting formulas — is derived under the assumption of weak stationarity. If you fit an ARMA to a non-stationary series, your coefficient estimates, standard errors, and forecasts are all invalid.

#### Intuition: what stationarity *looks like*

| Property | Stationary | Non-stationary |
|---|---|---|
| Mean over time | Flat, constant | Drifts upward/downward |
| Variance over time | Stable | Grows or changes |
| Correlation structure | Same for any window | Changes over time |
| Does it return to a level? | Yes (mean-reverting) | No |
| ACF plot | Decays to zero quickly | Decays very slowly (near 1) |

### 📈 Trading perspective

- **Equity prices** are generally I(1) — non-stationary in levels, stationary in log-returns. You cannot use an AR(1) on price levels and expect valid inferences.
- **Spreads** (e.g., 10Y Bund yield minus 2Y yield, or long/short equity pairs) may be stationary — this is the basis of **pairs trading** and **spread mean-reversion strategies**. Testing for stationarity of the spread is the first step (ADF test — Module 6).
- **FX rates** famously approximate random walks (Meese-Rogoff 1983 puzzle). The first difference (daily return) is approximately stationary.

In [ ]:
# ── Visual stationarity check: rolling mean and variance ─────────────────────
# A stationary series should have roughly flat rolling mean and variance.

window = 252  # 1 year of trading days

series = {
    'DAX level (non-stationary)':   dax_close,
    'DAX log-returns (stationary)': dax_ret,
}

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for row, (name, s) in enumerate(series.items()):
    roll_mean = s.rolling(window).mean()
    roll_std  = s.rolling(window).std()

    axes[row, 0].plot(s, color=BLUE, lw=0.6, alpha=0.7)
    axes[row, 0].plot(roll_mean, color=CORAL, lw=1.8, label=f'{window}d rolling mean')
    axes[row, 0].set_title(f'{name} — rolling mean')
    axes[row, 0].legend(fontsize=9)

    axes[row, 1].plot(roll_std, color=AMBER, lw=1.2)
    axes[row, 1].set_title(f'{name} — rolling std')

plt.suptitle('Rolling statistics: flat = consistent with stationarity', y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

print('Observation:')
print('  DAX LEVEL:   rolling mean trends upward, rolling std changes → NON-STATIONARY')
print('  DAX RETURNS: rolling mean ≈ 0, rolling std relatively stable → CONSISTENT WITH STATIONARITY')
print('  (Note: return volatility clusters → ARCH effects, not non-stationarity)')

In [ ]:
# ── Mean of AR, MA, and ARMA processes — formula verification ─────────────────
# Wohlrabe derives: E(y_t) = α₀ / (1 - α₁ - α₂ - ... - αₚ)  for AR(p) with constant
# This cell simulates processes and compares sample mean to the theoretical formula.

from statsmodels.tsa.arima_process import ArmaProcess

np.random.seed(42)
T = 5000

def arma_mean_formula(alpha0, ar_coeffs):
    """Theoretical mean: α₀ / (1 - Σαᵢ)"""
    return alpha0 / (1 - sum(ar_coeffs))

# Simulate AR(1): y_t = 0.5 + 0.7*y_{t-1} + ε_t   → mean = 0.5 / (1-0.7) = 1.667
# ArmaProcess uses: ar=[1, -α₁], ma=[1]
ar1_proc = ArmaProcess(ar=np.array([1, -0.7]), ma=np.array([1]))
ar1_sim  = 0.5 + ar1_proc.generate_sample(T)  # add constant as mean shift

# Simulate AR(2): y_t = 2 + 0.5*y_{t-1} + 0.2*y_{t-2} + ε_t  → mean = 2/(1-0.5-0.2) = 6.667
ar2_proc = ArmaProcess(ar=np.array([1, -0.5, -0.2]), ma=np.array([1]))
ar2_sim  = 2.0 + ar2_proc.generate_sample(T)  # constant adds to intercept effect

# MA(1): y_t = 3 + 0.8*ε_{t-1} + ε_t   → mean = β₀ = 3
ma1_proc = ArmaProcess(ar=np.array([1]), ma=np.array([1, 0.8]))
ma1_sim  = 3.0 + ma1_proc.generate_sample(T)

print(f'{"Process":<30} {"Theoretical mean":>20} {"Sample mean":>15} {"Difference":>12}')
print('-' * 80)

cases = [
    ('AR(1): α₀=0.5, α₁=0.7',  arma_mean_formula(0.5, [0.7]),   ar1_sim),
    ('AR(2): α₀=2.0, α₁=0.5, α₂=0.2', arma_mean_formula(2.0, [0.5, 0.2]), ar2_sim),
    ('MA(1): β₀=3.0',           3.0,                              ma1_sim),
]
for name, theory, sim in cases:
    sample = sim.mean()
    print(f'{name:<30} {theory:>20.4f} {sample:>15.4f} {abs(theory-sample):>12.4f}')

print('\nLaw of large numbers at work: sample means converge to theoretical means as T → ∞')

<a id='3'></a>
---
## 3 · AR, MA, and ARMA Processes — The Algebra

### 📖 Theory

#### AR(p) — Autoregressive process
$$y_t = \alpha_1 y_{t-1} + \ldots + \alpha_p y_{t-p} + \varepsilon_t \quad\Leftrightarrow\quad \alpha(L)y_t = \varepsilon_t$$
Mean: $\mu = \dfrac{\alpha_0}{1 - \alpha_1 - \ldots - \alpha_p}$

#### MA(q) — Moving average process
$$y_t = \varepsilon_t + \beta_1 \varepsilon_{t-1} + \ldots + \beta_q \varepsilon_{t-q} \quad\Leftrightarrow\quad y_t = \beta(L)\varepsilon_t$$
Mean: $\mu = \beta_0$ (the constant).  **MA processes are always stationary** — they are finite sums of white noise.

#### ARMA(p,q) — Mixed process
$$y_t = \alpha_1 y_{t-1} + \ldots + \alpha_p y_{t-p} + \varepsilon_t + \beta_1 \varepsilon_{t-1} + \ldots + \beta_q \varepsilon_{t-q}$$
$$\alpha(L)y_t = \beta(L)\varepsilon_t$$

Stationarity depends **only** on the AR part ($\alpha$ polynomial). The MA parameters do not affect stationarity.

#### ARIMA(p,d,q) — Integrated
$$\alpha(L)\Delta^d y_t = \beta(L)\varepsilon_t$$

Applied to the $d$-th difference of the original series. For financial prices: $d=1$ (returns). For GDP growth (already differenced): $d=0$.

#### The AR(1) → MA(∞) connection

By recursive substitution, any stationary AR(1) with $|\alpha| < 1$:
$$y_t = \sum_{j=0}^{\infty} \alpha^j \varepsilon_{t-j}$$

This is an MA($\infty$) representation. Every stationary AR process has one — this is the Wold theorem (Section 9).

In [ ]:
# ── Simulate AR(1) with different α values — Wohlrabe's slide 161 ─────────────
# α controls persistence: near 0 = rapid mean reversion, near 1 = slow, =1 = unit root

np.random.seed(99)
T = 200
eps = np.random.normal(0, 1, T)

def simulate_ar1(alpha, T, eps):
    y = np.zeros(T)
    for t in range(1, T):
        y[t] = alpha * y[t-1] + eps[t]
    return y

alphas = [0.3, 0.7, 0.95, 1.0, 1.05]
labels = ['α=0.3 (fast reversion)', 'α=0.7 (moderate)', 'α=0.95 (persistent)',
          'α=1.0 (unit root = RW)', 'α=1.05 (explosive)']
colors = [TEAL, BLUE, AMBER, CORAL, PINK]

fig, axes = plt.subplots(1, 5, figsize=(18, 4), sharey=False)
for i, (a, lbl, col) in enumerate(zip(alphas, labels, colors)):
    y = simulate_ar1(a, T, eps)
    axes[i].plot(y, color=col, lw=0.9)
    axes[i].axhline(0, color=GRAY, lw=0.5)
    axes[i].set_title(lbl, fontsize=9)

plt.suptitle('AR(1) behaviour across α values — same shock sequence', y=1.02, fontsize=11)
plt.tight_layout()
plt.show()

print('Key observations:')
print('  α = 0.3: rapidly reverts to 0 (short memory)')
print('  α = 0.95: slow reversion — looks like a trend but is stationary')
print('  α = 1.0: never reverts (random walk, non-stationary)')
print('  α > 1.0: explosive — variance → ∞ (impossible for real assets over time)')

In [ ]:
# ── AR(1) → MA(∞): verify the recursive representation ───────────────────────
# y_t = Σ αʲ ε_{t-j}  for j=0,1,2,...
# Truncate at K=50 terms and compare to direct simulation

np.random.seed(42)
T, K = 300, 50
alpha = 0.7
eps = np.random.normal(0, 1, T + K)

# Direct AR simulation
y_ar = np.zeros(T + K)
for t in range(1, T + K):
    y_ar[t] = alpha * y_ar[t-1] + eps[t]
y_ar = y_ar[K:]  # discard burn-in

# MA(∞) approximation: y_t = Σ_{j=0}^{K} αʲ ε_{t-j}
y_ma_inf = np.zeros(T)
eps_used = eps  # length T+K
for t in range(T):
    y_ma_inf[t] = sum(alpha**j * eps_used[K + t - j] for j in range(K+1))

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(y_ar,     color=BLUE,  lw=1.2, label='AR(1) direct simulation')
ax.plot(y_ma_inf, color=CORAL, lw=1.0, ls='--', label=f'MA(∞) truncated at K={K}')
ax.set_title(f'AR(1) with α={alpha}: direct simulation vs MA(∞) representation')
ax.legend()
plt.tight_layout()
plt.show()

corr = np.corrcoef(y_ar, y_ma_inf)[0,1]
print(f'Correlation between AR(1) and MA(∞) truncated representation: {corr:.6f}')
print(f'(Should be ≈ 1.0; discrepancy comes from truncating the infinite sum at K={K})')

<a id='4'></a>
---
## 4 · Stationarity Conditions for AR(p)

### 📖 Theory

An AR(p) process is **covariance stationary** if and only if **all roots of the characteristic polynomial lie strictly outside the unit circle**:

$$\alpha(\lambda) = 1 - \alpha_1 \lambda - \alpha_2 \lambda^2 - \ldots - \alpha_p \lambda^p = 0 \implies |\lambda_i| > 1 \quad \forall i$$

For the most common cases:

**AR(1):** Stationary $\Leftrightarrow$ $|\alpha_1| < 1$

**AR(2):** Stationary $\Leftrightarrow$ simultaneously:
$$|\alpha_2| < 1 \qquad \alpha_2 + \alpha_1 < 1 \qquad \alpha_2 - \alpha_1 < 1$$

These three conditions define a triangular region in $(\alpha_1, \alpha_2)$ space — the **stationarity triangle**.

For MA(q): **always stationary** (it is a finite sum of white noise terms, each with finite variance).

For ARMA(p,q): stationarity depends **only** on the AR part; the MA parameters are irrelevant for stationarity.

### 📈 Trading perspective
When you estimate an ARMA model on a spread series (say, the 10Y–2Y Bund yield spread), you should always verify that the fitted AR coefficients satisfy the stationarity conditions. If they don't, your model is misspecified — the spread would be explosive in your model, which makes no economic sense.

In [ ]:
# ── AR(2) stationarity triangle ───────────────────────────────────────────────
# The set of (α₁, α₂) pairs for which AR(2) is stationary forms a triangle.
# Conditions: |α₂| < 1,  α₁+α₂ < 1,  α₂-α₁ < 1

a1 = np.linspace(-2.5, 2.5, 400)
a2 = np.linspace(-1.5, 1.5, 400)
A1, A2 = np.meshgrid(a1, a2)

# Stationarity: all three inequalities must hold
stationary = (np.abs(A2) < 1) & (A1 + A2 < 1) & (A2 - A1 < 1)

fig, ax = plt.subplots(figsize=(7, 6))
ax.contourf(A1, A2, stationary.astype(float), levels=[0.5, 1.5],
            colors=[TEAL], alpha=0.3)
ax.contour(A1, A2, stationary.astype(float), levels=[0.5], colors=[TEAL], linewidths=2)

# Mark some interesting points
points = {
    'Stationary (0.5, 0.2)':    (0.5,  0.2,  TEAL),
    'Borderline (1.0, 0.0)':    (1.0,  0.0,  AMBER),
    'Non-stationary (1.2, 0.3)':(1.2,  0.3,  CORAL),
    'Oscillating (-1.5, 0.5)':  (-1.5, 0.5,  BLUE),
}
for label, (x, y, col) in points.items():
    ax.scatter(x, y, color=col, s=80, zorder=5)
    ax.annotate(label, (x, y), xytext=(10, 5), textcoords='offset points', fontsize=8, color=col)

ax.axhline(0, color=GRAY, lw=0.5)
ax.axvline(0, color=GRAY, lw=0.5)
ax.set_xlabel('$\\alpha_1$')
ax.set_ylabel('$\\alpha_2$')
ax.set_title('AR(2) stationarity region (shaded = stationary)')
plt.tight_layout()
plt.show()

print('The triangle is bounded by:')
print('  Bottom: α₂ > -1')
print('  Top-right: α₁ + α₂ < 1')
print('  Top-left:  α₂ - α₁ < 1')

In [ ]:
# ── Real-world check: fit AR(2) to German Bund yield spread, verify stationarity
# The 10Y-2Y Bund spread is a classic macro signal (yield curve steepness)

bund_10y = fred.get_series('IRLTLT01DEM156N', observation_start='2000-01-01')  # 10Y DE
bund_2y  = fred.get_series('IRSTCI01DEM156N', observation_start='2000-01-01')  # 2Y DE

spread = (bund_10y - bund_2y).dropna()
spread.name = 'Bund 10Y-2Y spread (pp)'

# Fit AR(2)
ar2_model = ARIMA(spread, order=(2, 0, 0), trend='c').fit()
alpha1_hat = ar2_model.params['ar.L1']
alpha2_hat = ar2_model.params['ar.L2']

# Check stationarity of fitted AR(2)
roots = char_roots([alpha1_hat, alpha2_hat])
mods  = np.abs(roots)
is_stat = np.all(mods > 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(spread, color=BLUE, lw=1)
axes[0].axhline(spread.mean(), color=CORAL, ls='--', lw=1.5, label=f'Mean = {spread.mean():.2f} pp')
axes[0].set_title('Germany: Bund 10Y–2Y yield spread')
axes[0].set_ylabel('Percentage points')
axes[0].legend()

# Root plot
theta = np.linspace(0, 2*np.pi, 300)
axes[1].plot(np.cos(theta), np.sin(theta), color=GRAY, lw=1.5, label='Unit circle')
axes[1].scatter(roots.real, roots.imag, color=CORAL, s=100, zorder=5, label='Characteristic roots')
# Stationarity requires roots outside the unit circle → 1/root inside unit circle
inv_roots = 1/roots
axes[1].scatter(inv_roots.real, inv_roots.imag, color=TEAL, s=100, marker='x',
                zorder=5, label='Inverse roots (should be inside circle)')
axes[1].axhline(0, color=GRAY, lw=0.5)
axes[1].axvline(0, color=GRAY, lw=0.5)
axes[1].set_aspect('equal')
axes[1].set_title('Characteristic roots of fitted AR(2)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f'Fitted AR(2):  α₁ = {alpha1_hat:.4f},  α₂ = {alpha2_hat:.4f}')
print(f'Characteristic root moduli: {mods[0]:.4f}, {mods[1]:.4f}')
print(f'All roots |λ| > 1:  {"YES — process is stationary" if is_stat else "NO — non-stationary"}')

<a id='5'></a>
---
## 5 · Causality and Invertibility

### 📖 Theory

Two additional properties are required for ARMA models to be well-behaved:

#### Causality
An ARMA process is **causal** if the current value $y_t$ can be expressed as a one-sided MA($\infty$) in **past and present** innovations only:
$$y_t = \sum_{j=0}^{\infty} \psi_j \varepsilon_{t-j}, \qquad \sum_{j=0}^{\infty} |\psi_j| < \infty$$

**Equivalent condition:** All roots of the AR polynomial $\alpha(\lambda) = 0$ lie **outside the unit circle**.

This is the same as the stationarity condition for AR processes. A causal process depends only on current and past shocks — sensible for any real-world time series.

#### Invertibility
An ARMA process is **invertible** if the current innovation $\varepsilon_t$ can be expressed as a one-sided AR($\infty$) in **past and present** observations:
$$\varepsilon_t = \sum_{j=0}^{\infty} \pi_j y_{t-j}, \qquad \sum_{j=0}^{\infty} |\pi_j| < \infty$$

**Equivalent condition:** All roots of the MA polynomial $\beta(\lambda) = 0$ lie **outside the unit circle**.

**Why invertibility matters:** Invertibility guarantees a unique MA representation. Without it, the same ARMA model could have multiple equivalent MA forms, making estimation unreliable. It is the condition that ties the observed innovations to past data.

**Summary table:**

| Condition | Which polynomial? | Root requirement | Consequence |
|---|---|---|---|
| Causality | AR: $\alpha(\lambda)$ | All roots outside unit circle | $y_t$ depends only on past $\varepsilon$ |
| Invertibility | MA: $\beta(\lambda)$ | All roots outside unit circle | $\varepsilon_t$ recoverable from past $y$ |

In [ ]:
# ── Check causality and invertibility for several ARMA models ────────────────
# Replicating Wohlrabe's examples (slides 193–195)

def check_arma(ar_coeffs, ma_coeffs, name):
    """
    Check causality (AR roots outside unit circle)
    and invertibility (MA roots outside unit circle).
    Convention: ar_coeffs = [α₁, α₂, ...], ma_coeffs = [β₁, β₂, ...]
    """
    def get_roots(coeffs):
        if not coeffs:
            return np.array([])
        poly = np.array([1] + [-c for c in coeffs])
        return np.roots(poly[::-1])

    ar_roots = get_roots(ar_coeffs)
    ma_roots = get_roots(ma_coeffs)

    causal    = len(ar_roots) == 0 or np.all(np.abs(ar_roots) > 1)
    invertible = len(ma_roots) == 0 or np.all(np.abs(ma_roots) > 1)

    ar_str = ', '.join([f'|λ|={abs(r):.3f}' for r in ar_roots]) if len(ar_roots) else 'pure MA'
    ma_str = ', '.join([f'|λ|={abs(r):.3f}' for r in ma_roots]) if len(ma_roots) else 'pure AR'

    print(f'\n{name}')
    print(f'  AR roots: {ar_str}  → Causal:      {"YES ✓" if causal     else "NO  ✗"}')
    print(f'  MA roots: {ma_str}  → Invertible:  {"YES ✓" if invertible else "NO  ✗"}')
    return causal, invertible

print('Causality and invertibility checks:')
print('=' * 60)
check_arma([1.5], [0.2],    'AR(1,ARMA): y_t = 1.5y_{t-1} + 0.2ε_{t-1} + ε_t   [NOT causal]')
check_arma([0.5], [2.0],    'ARMA(1,1):  y_t = 0.5y_{t-1} + 2.0ε_{t-1} + ε_t   [NOT invertible]')
check_arma([0.7], [0.4],    'ARMA(1,1):  y_t = 0.7y_{t-1} + 0.4ε_{t-1} + ε_t   [Both OK]')
check_arma([0.0, 0.25], [2.0], 'ARMA(2,1): y_t = 0.25y_{t-2} + 2ε_{t-1} + ε_t  [NOT invertible]')
check_arma([0.5, 0.3], [0.4],  'ARMA(2,1): y_t = 0.5y_{t-1}+0.3y_{t-2}+0.4ε_{t-1}+ε_t [Both OK]')

<a id='6'></a>
---
## 6 · The Autocovariance Function (ACVF) — Derivation

### 📖 Theory

The **autocovariance function** for a stationary process is:
$$\gamma(h) = \text{Cov}(y_t, y_{t-h}) = E[(y_t - \mu)(y_{t-h} - \mu)]$$

Properties: $\gamma(0) \geq 0$, $|\gamma(h)| \leq \gamma(0)$, $\gamma(h) = \gamma(-h)$ (symmetric).

#### AR(1) autocovariance — full derivation (Wohlrabe slides 197–202)

For $y_t = \alpha y_{t-1} + \varepsilon_t$ with $|\alpha| < 1$:

**Step 1 — Variance:** Multiply both sides by $y_t$, take expectations:
$$\gamma(0) = \alpha \gamma(1) + \sigma^2 \qquad (\text{since } E(\varepsilon_t y_t) = \sigma^2)$$

**Step 2 — First autocovariance:** Multiply by $y_{t-1}$, take expectations:
$$\gamma(1) = \alpha \gamma(0) \qquad (\text{since } E(\varepsilon_t y_{t-1}) = 0)$$

**Step 3 — Solve:** Substitute $\gamma(1) = \alpha \gamma(0)$ into the variance equation:
$$\gamma(0) = \frac{\sigma^2}{1-\alpha^2}$$

**Step 4 — General lag:** By induction, $\gamma(h) = \alpha^h \gamma(0) = \dfrac{\alpha^h \sigma^2}{1-\alpha^2}$

#### ACF of AR(1)
$$\rho(h) = \frac{\gamma(h)}{\gamma(0)} = \alpha^h \qquad h = 0, 1, 2, \ldots$$

This is geometric decay. The **sign of $\alpha$** determines whether it decays monotonically ($\alpha > 0$) or oscillates ($\alpha < 0$).

#### ARMA(1,1) autocovariance — key result
For $y_t = \alpha y_{t-1} + \varepsilon_t + \beta \varepsilon_{t-1}$:
$$\gamma(0) = \frac{(1 + 2\alpha\beta + \beta^2)\sigma^2}{1-\alpha^2}, \qquad \gamma(1) = \frac{(1+\alpha\beta)(\alpha+\beta)\sigma^2}{1-\alpha^2}$$
$$\gamma(h) = \alpha^{h-1}\gamma(1) \quad \text{for } h \geq 2 \qquad (\text{same geometric decay as AR(1) for large } h)$$

In [ ]:
# ── Theoretical vs sample ACVF for AR(1) ─────────────────────────────────────

def ar1_theoretical_acvf(alpha, sigma2, max_lag):
    """Compute theoretical ACVF for AR(1): γ(h) = α^h * σ²/(1-α²)"""
    gamma0 = sigma2 / (1 - alpha**2)
    return np.array([alpha**h * gamma0 for h in range(max_lag+1)])

np.random.seed(42)
T      = 2000
alpha  = 0.75
sigma2 = 1.0
max_lag = 20

ar1_proc = ArmaProcess(ar=np.array([1, -alpha]), ma=np.array([1]))
y_sim    = ar1_proc.generate_sample(T, scale=np.sqrt(sigma2))

# Sample ACVF
sample_acvf = np.array([np.cov(y_sim[h:], y_sim[:-h] if h > 0 else y_sim)[0,1]
                         if h > 0 else np.var(y_sim) for h in range(max_lag+1)])

# Theoretical ACVF
theory_acvf = ar1_theoretical_acvf(alpha, sigma2, max_lag)

lags = np.arange(max_lag+1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(lags, theory_acvf, 'o-', color=CORAL, lw=2, ms=6, label='Theoretical γ(h)')
axes[0].bar(lags, sample_acvf, color=BLUE, alpha=0.4, label='Sample ACVF')
axes[0].set_xlabel('Lag h')
axes[0].set_ylabel('Autocovariance')
axes[0].set_title(f'AR(1) ACVF: theoretical vs sample  (α={alpha}, T={T})')
axes[0].legend()

# ACF (normalized)
theory_acf = theory_acvf / theory_acvf[0]
sample_acf_ = sample_acvf / sample_acvf[0]
axes[1].plot(lags, theory_acf,  'o-', color=CORAL, lw=2, ms=6, label='Theoretical ACF = αʰ')
axes[1].bar(lags, sample_acf_,  color=BLUE, alpha=0.4, label='Sample ACF')
# 95% confidence band for sample ACF: ±1.96/√T
ci = 1.96 / np.sqrt(T)
axes[1].axhline( ci, color=GRAY, ls='--', lw=0.8, label='95% CI bands (±1.96/√T)')
axes[1].axhline(-ci, color=GRAY, ls='--', lw=0.8)
axes[1].axhline(0, color=GRAY, lw=0.4)
axes[1].set_xlabel('Lag h')
axes[1].set_ylabel('ACF')
axes[1].set_title(f'AR(1) ACF: ρ(h) = α^h = {alpha}^h')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f'Theoretical variance γ(0) = σ²/(1-α²) = {sigma2/(1-alpha**2):.4f}')
print(f'Sample variance                        = {np.var(y_sim):.4f}')

<a id='7'></a>
---
## 7 · The ACF in Practice — Identification Tool

### 📖 Theory

The **sample ACF** is the empirical counterpart:
$$\hat{\rho}(h) = \frac{\hat{\gamma}(h)}{\hat{\gamma}(0)}, \qquad \hat{\gamma}(h) = \frac{1}{T}\sum_{t=h+1}^{T}(y_t - \bar{y})(y_{t-h} - \bar{y})$$

Under the null that $\rho(h) = 0$, the sample ACF is approximately $N(0, 1/T)$. The **95% confidence band** is $\pm 1.96/\sqrt{T}$.

#### ACF patterns — the identification cheat sheet

| Process | ACF pattern | PACF pattern |
|---|---|---|
| White noise | All lags ≈ 0 | All lags ≈ 0 |
| AR(p) | Tails off (geometric/oscillating decay) | **Cuts off after lag p** |
| MA(q) | **Cuts off after lag q** | Tails off |
| ARMA(p,q) | Tails off (after lag q) | Tails off (after lag p) |
| I(1) (random walk) | Very slow decay, stays near 1 | First lag ≈ 1, rest ≈ 0 |

"Cuts off" means the ACF/PACF is exactly zero beyond a certain lag (in theory). In practice, spikes that fall within the 95% CI bands are treated as zero.

### 📈 Trading perspective
- **Return ACF ≈ 0** → little linear predictability in prices (consistent with weak EMH)
- **Squared return ACF ≠ 0** → volatility clustering is predictable → GARCH models (Module 7)
- **Volume or order flow ACF** often shows significant structure → some HFT strategies exploit this

In [ ]:
# ── ACF patterns for all process types — the identification cheat sheet ───────

np.random.seed(42)
T = 1000

processes = [
    ('White noise',   ArmaProcess([1],         [1]),          'WN'),
    ('AR(1) α=0.7',   ArmaProcess([1, -0.7],   [1]),          'AR1_pos'),
    ('AR(1) α=-0.7',  ArmaProcess([1,  0.7],   [1]),          'AR1_neg'),
    ('AR(2)',          ArmaProcess([1, -0.6, -0.3], [1]),      'AR2'),
    ('MA(1) β=0.7',   ArmaProcess([1],         [1,  0.7]),    'MA1'),
    ('MA(2)',          ArmaProcess([1],         [1, 0.6, 0.3]),'MA2'),
    ('ARMA(1,1)',      ArmaProcess([1, -0.6],   [1, 0.4]),     'ARMA11'),
]

fig = plt.figure(figsize=(18, 14))
for i, (name, proc, _) in enumerate(processes):
    y = proc.generate_sample(T)
    ax = fig.add_subplot(7, 2, 2*i+1)
    plot_acf(y, lags=20, ax=ax, color=BLUE, alpha=0.6)
    ax.set_title(f'{name} — ACF', fontsize=9)
    ax.set_ylim(-0.5, 1.1)

    ax2 = fig.add_subplot(7, 2, 2*i+2)
    plot_pacf(y, lags=20, ax=ax2, method='ywm', color=CORAL, alpha=0.6)
    ax2.set_title(f'{name} — PACF', fontsize=9)
    ax2.set_ylim(-0.5, 1.1)

plt.suptitle('ACF and PACF patterns — the Box-Jenkins identification guide', y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Apply to real data: DAX returns and EUR/USD returns ───────────────────────

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

series_list = [
    (dax_ret,     'DAX daily log-returns'),
    (eurusd_ret,  'EUR/USD daily log-returns'),
]

for row, (s, name) in enumerate(series_list):
    axes[row, 0].plot(s, color=BLUE, lw=0.5, alpha=0.7)
    axes[row, 0].axhline(0, color=GRAY, lw=0.5)
    axes[row, 0].set_title(name)

    plot_acf(s, lags=40, ax=axes[row, 1], color=TEAL, alpha=0.7)
    axes[row, 1].set_title(f'ACF — {name}')
    axes[row, 1].set_ylim(-0.15, 0.15)

    plot_acf(s**2, lags=40, ax=axes[row, 2], color=CORAL, alpha=0.7)
    axes[row, 2].set_title(f'ACF — Squared returns (ARCH signal)')

plt.tight_layout()
plt.show()

# Ljung-Box test: are any ACF spikes significant?
from statsmodels.stats.diagnostic import acorr_ljungbox

for s, name in series_list:
    lb_ret = acorr_ljungbox(s,    lags=[10, 20], return_df=True)
    lb_sq  = acorr_ljungbox(s**2, lags=[10, 20], return_df=True)
    print(f'\n{name}')
    print(f'  Ljung-Box test on returns (p-values):')
    print(f'    lag=10: p={lb_ret["lb_pvalue"].iloc[0]:.4f}  lag=20: p={lb_ret["lb_pvalue"].iloc[1]:.4f}')
    print(f'  Ljung-Box test on SQUARED returns:')
    print(f'    lag=10: p={lb_sq["lb_pvalue"].iloc[0]:.2e}  lag=20: p={lb_sq["lb_pvalue"].iloc[1]:.2e}')
    print(f'  → Returns: {"some autocorrelation" if lb_ret["lb_pvalue"].min() < 0.05 else "approximately white noise"}')
    print(f'  → Squared returns: {"strong ARCH effects (use GARCH)" if lb_sq["lb_pvalue"].min() < 0.05 else "no ARCH effects"}')

<a id='8'></a>
---
## 8 · The Partial Autocorrelation Function (PACF)

### 📖 Theory

The **partial autocorrelation** at lag $h$, written $\phi_{hh}$, measures the correlation between $y_t$ and $y_{t-h}$ **after removing the linear effect of all intermediate lags** $y_{t-1}, \ldots, y_{t-h+1}$.

Formally, $\phi_{hh}$ is the last coefficient in the best linear predictor of $y_t$ given $y_{t-1}, \ldots, y_{t-h}$:
$$\hat{y}_t = \phi_{h1} y_{t-1} + \phi_{h2} y_{t-2} + \ldots + \phi_{hh} y_{t-h}$$

The PACF is computed recursively via the **Durbin-Levinson algorithm** using the Yule-Walker equations:

$$\phi_{11} = \rho(1)$$
$$\phi_{hh} = \frac{\rho(h) - \sum_{k=1}^{h-1} \phi_{h-1,k}\, \rho(h-k)}{1 - \sum_{k=1}^{h-1} \phi_{h-1,k}\, \rho(k)}$$

#### Identifying AR order from the PACF

For an AR(p) process, the PACF **cuts off exactly at lag $p$**: $\phi_{hh} = 0$ for $h > p$. This is the key identification result:
- **AR(1):** PACF has one spike at lag 1, then all zero
- **AR(2):** PACF has spikes at lags 1 and 2, then zero
- **MA(q):** PACF tails off (no clean cutoff)

The 95% confidence band for the PACF is also $\pm 1.96/\sqrt{T}$.

In [ ]:
# ── PACF: Yule-Walker manual computation vs statsmodels ──────────────────────

def pacf_yule_walker(acf_values, max_lag):
    """
    Compute PACF via Durbin-Levinson recursion.
    acf_values: [ρ(0), ρ(1), ρ(2), ...]
    Returns PACF values [φ₁₁, φ₂₂, ..., φ_{max_lag, max_lag}]
    """
    rho = acf_values
    phi = {}  # phi[h][k] = φ_{hk}
    pacf_vals = []

    # h=1
    phi[1] = {1: rho[1]}
    pacf_vals.append(rho[1])

    for h in range(2, max_lag+1):
        prev = phi[h-1]
        num = rho[h] - sum(prev[k] * rho[h-k] for k in range(1, h))
        den = 1       - sum(prev[k] * rho[k]   for k in range(1, h))
        phi_hh = num / den
        pacf_vals.append(phi_hh)
        phi[h] = {k: prev[k] - phi_hh * prev[h-k] for k in range(1, h)}
        phi[h][h] = phi_hh

    return np.array(pacf_vals)

# Simulate AR(3) — PACF should cut off at lag 3
np.random.seed(7)
ar3_proc = ArmaProcess(ar=np.array([1, -0.5, -0.2, -0.1]), ma=np.array([1]))
y_ar3    = ar3_proc.generate_sample(2000)

# Get sample ACF
sample_acf_vals = acf(y_ar3, nlags=20)

# Manual Yule-Walker PACF
pacf_manual = pacf_yule_walker(sample_acf_vals, max_lag=20)

# Statsmodels PACF
pacf_sm = pacf(y_ar3, nlags=20, method='ywm')[1:]  # drop lag 0

ci = 1.96 / np.sqrt(len(y_ar3))
lags_ = np.arange(1, 21)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(lags_, pacf_manual, color=BLUE, alpha=0.6, label='Manual (Yule-Walker)')
axes[0].plot(lags_, pacf_sm, 'o', color=CORAL, ms=5, label='statsmodels PACF')
axes[0].axhline( ci, color=GRAY, ls='--', lw=0.8)
axes[0].axhline(-ci, color=GRAY, ls='--', lw=0.8)
axes[0].axhline(0,   color=GRAY, lw=0.4)
axes[0].set_title('PACF of simulated AR(3) — should cut off at lag 3')
axes[0].set_xlabel('Lag h')
axes[0].legend()

# Confirm: add ACF next to it
axes[1].bar(np.arange(1, 21), sample_acf_vals[1:], color=TEAL, alpha=0.6)
axes[1].axhline( ci, color=GRAY, ls='--', lw=0.8)
axes[1].axhline(-ci, color=GRAY, ls='--', lw=0.8)
axes[1].axhline(0, color=GRAY, lw=0.4)
axes[1].set_title('ACF of simulated AR(3) — tails off geometrically')
axes[1].set_xlabel('Lag h')

plt.tight_layout()
plt.show()

print('PACF values at lags 1–5:')
for lag in range(1, 6):
    print(f'  φ_{lag}{lag} = {pacf_manual[lag-1]:.4f}  (significant: {"YES" if abs(pacf_manual[lag-1]) > ci else "no "})')

In [ ]:
# ── PACF applied to real macro data: what AR order does German IP suggest? ────

ip_sa = fred.get_series('DEUPROINDMISMEI', observation_start='1991-01-01').dropna()
# Take log-differences (monthly growth rates)
ip_growth = np.log(ip_sa).diff().dropna() * 100
ip_growth.name = 'German IP growth (monthly %, log-diff)'

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(ip_growth, color=TEAL, lw=0.8)
axes[0].axhline(0, color=GRAY, lw=0.5)
axes[0].set_title('German Industrial Production — monthly log-growth')
axes[0].set_ylabel('%')

plot_acf(ip_growth, lags=24, ax=axes[1], color=BLUE)
axes[1].set_title('ACF — German IP growth')

plot_pacf(ip_growth, lags=24, ax=axes[2], method='ywm', color=CORAL)
axes[2].set_title('PACF — German IP growth\n(significant spikes → suggested AR order)')

plt.tight_layout()
plt.show()

# Get top PACF spikes
pacf_vals = pacf(ip_growth, nlags=24, method='ywm')
ci_ip = 1.96 / np.sqrt(len(ip_growth))
sig_lags = [h for h in range(1, 25) if abs(pacf_vals[h]) > ci_ip]
print(f'Statistically significant PACF lags (95% level): {sig_lags}')
print(f'→ Suggests AR order p ≥ {max(sig_lags) if sig_lags else 0}')
print(f'(Also check for seasonal spikes at lags 12, 24 — common for monthly industrial data)')

<a id='9'></a>
---
## 9 · The Wold Decomposition Theorem

### 📖 Theory

The **Wold Decomposition Theorem** (Wold, 1938) is the theoretical foundation that justifies using ARMA models for any covariance-stationary process.

> **Theorem (Wold):** Every covariance-stationary process $\{y_t\}$ can be written as the sum of two orthogonal components:
> $$y_t = \underbrace{\sum_{j=0}^{\infty} \psi_j \varepsilon_{t-j}}_{\text{linearly indeterministic}} + \underbrace{\kappa_t}_{\text{linearly deterministic}}$$
> where:
> - $\varepsilon_t = y_t - P_{t-1}y_t$ is the **innovation** (the unpredictable part)
> - $\{\varepsilon_t\}$ is white noise with $\text{Var}(\varepsilon_t) = \sigma^2$
> - $\sum_{j=0}^{\infty} \psi_j^2 < \infty$ (square-summable weights)
> - $\kappa_t$ is perfectly predictable from its own past (e.g., a seasonal dummy, a deterministic trend)

**What this means in practice:**
- Any stationary time series has an MA($\infty$) representation (the first component)
- ARMA(p,q) is a *parsimonious approximation* to the infinite MA representation
- The residual after fitting an ARMA model should approximate $\varepsilon_t$ — the Wold innovation

**Why it's important for trading:**
- It guarantees that if a return series is stationary, it has predictable structure (even if small)
- The innovation $\varepsilon_t$ represents the component that is genuinely unforecastable — the irreducible uncertainty
- The $\psi_j$ coefficients are the **impulse response function** — how a shock today propagates through time

#### Impulse Response Function (IRF)
$$\frac{\partial y_{t+h}}{\partial \varepsilon_t} = \psi_h$$

For AR(1): $\psi_h = \alpha^h$ — a shock decays at rate $\alpha$ per period. For $\alpha = 0.9$, a unit shock at $t=0$ still has $0.9^{10} \approx 35\%$ of its effect after 10 periods.

In [ ]:
# ── Wold representation: impulse response functions ───────────────────────────
# The ψ_j weights show how a one-unit shock today propagates into the future.

def impulse_response(ar_params, ma_params, periods=30):
    """
    Compute impulse response function for ARMA(p,q).
    ar_params: [α₁, α₂, ...] (without leading 1)
    ma_params: [β₁, β₂, ...] (without leading 1)
    """
    # Use ArmaProcess: arparams includes leading 1 with negation
    ar = np.r_[1, -np.array(ar_params)]
    ma = np.r_[1,  np.array(ma_params)]
    proc = ArmaProcess(ar, ma)
    # Impulse response = MA representation coefficients
    return proc.impulse_response(periods)

models = {
    'AR(1) α=0.3  (fast decay)':   ([0.3],      []),
    'AR(1) α=0.7  (moderate)':     ([0.7],      []),
    'AR(1) α=0.95 (very persistent)': ([0.95],  []),
    'MA(1) β=0.7  (1-period shock)':  ([],      [0.7]),
    'ARMA(1,1) α=0.7, β=0.4':     ([0.7],      [0.4]),
}
colors_ = [TEAL, BLUE, AMBER, PINK, CORAL]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for (name, (ar, ma)), col in zip(models.items(), colors_):
    irf = impulse_response(ar, ma, periods=30)
    axes[0].plot(irf, color=col, lw=1.5, label=name)

axes[0].axhline(0, color=GRAY, lw=0.5)
axes[0].set_title('Impulse Response Functions  ($\\psi_h$)\n= response of $y_{t+h}$ to a unit shock at $t$')
axes[0].set_xlabel('Horizon h (periods)')
axes[0].set_ylabel('Response $\\psi_h$')
axes[0].legend(fontsize=8)

# Cumulative IRF: total effect of a permanent shock
for (name, (ar, ma)), col in zip(models.items(), colors_):
    irf = impulse_response(ar, ma, periods=30)
    axes[1].plot(np.cumsum(irf), color=col, lw=1.5, label=name)

axes[1].axhline(0, color=GRAY, lw=0.5)
axes[1].set_title('Cumulative IRF  ($\\sum_{j=0}^h \\psi_j$)\n= long-run multiplier')
axes[1].set_xlabel('Horizon h (periods)')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print('Trading interpretation of IRF:')
print('  AR(1) α=0.95: a price shock has 95% of its effect 1 period later.')
print('                After 20 days: 0.95^20 ≈ {:.1%} of original shock remains.'.format(0.95**20))
print('  MA(1) β=0.7:  a shock has a direct effect and one delayed effect, then gone.')
print('  ARMA(1,1):    the MA part amplifies/dampens the initial response.')

<a id='10'></a>
---
## 10 · Ergodicity — Why Sample Moments Work

### 📖 Theory

We have only **one realization** of the stochastic process — one path of history. So how can we estimate the mean and autocovariances of the underlying process?

The answer is **ergodicity**: for an ergodic process, time averages converge to ensemble (population) averages:

$$\bar{y} = \frac{1}{T}\sum_{t=1}^T y_t \xrightarrow{p} E(y_t) = \mu \quad \text{as } T \to \infty$$

$$\hat{\gamma}(h) = \frac{1}{T}\sum_{t=h+1}^T (y_t - \bar{y})(y_{t-h} - \bar{y}) \xrightarrow{p} \gamma(h)$$

This is the **law of large numbers for time series** — it replaces the usual i.i.d. assumption.

**Sufficient condition for ergodicity:** $\sum_{h=0}^{\infty} |\gamma(h)| < \infty$, i.e., autocorrelations decay fast enough.

**What breaks ergodicity:** A non-stationary process (like a random walk) is not ergodic — the time average does not converge. If you compute the sample mean of a random walk, it just tells you where the walk happened to end up, not the "true" mean (which is undefined).

### 📈 Trading perspective
Ergodicity is the silent assumption behind every backtesting exercise. When you compute average returns or Sharpe ratios over a historical period, you are assuming the history is representative of the underlying distribution. If the process has structural breaks (e.g., the 2008 regime shift), ergodicity fails and your backtest is misleading.

In [ ]:
# ── Ergodicity in action: sample mean convergence for stationary vs RW ────────

np.random.seed(0)
T    = 2000
N    = 30    # number of independent paths

# True mean for AR(1) with intercept: y_t = 1 + 0.7*y_{t-1} + ε_t  → μ = 1/(1-0.7) = 3.33
true_mean = 1.0 / (1 - 0.7)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for trial in range(N):
    eps = np.random.normal(0, 1, T)

    # AR(1): stationary
    y_ar1 = np.zeros(T)
    for t in range(1, T):
        y_ar1[t] = 1 + 0.7 * y_ar1[t-1] + eps[t]
    cum_mean_ar1 = np.cumsum(y_ar1) / np.arange(1, T+1)

    # Random walk: non-stationary, non-ergodic
    y_rw = np.cumsum(eps)
    cum_mean_rw = np.cumsum(y_rw) / np.arange(1, T+1)

    alpha = 0.15 if trial < N-1 else 0.9
    col_a = BLUE  if trial < N-1 else TEAL
    col_r = CORAL if trial < N-1 else PINK

    axes[0].plot(cum_mean_ar1, color=col_a, lw=0.5, alpha=alpha)
    axes[1].plot(cum_mean_rw,  color=col_r, lw=0.5, alpha=alpha)

axes[0].axhline(true_mean, color=CORAL, lw=2, ls='--', label=f'True mean = {true_mean:.2f}')
axes[0].set_title(f'AR(1) — cumulative mean across {N} paths\n(ergodic: all converge to μ)')
axes[0].set_xlabel('T (sample size)')
axes[0].set_ylabel('Running sample mean')
axes[0].legend()

axes[1].axhline(0, color=GRAY, lw=1.5, ls='--', label='"True" mean = 0')
axes[1].set_title(f'Random walk — cumulative mean across {N} paths\n(non-ergodic: paths diverge)')
axes[1].set_xlabel('T (sample size)')
axes[1].set_ylabel('Running sample mean')
axes[1].legend()

plt.tight_layout()
plt.show()

print('Ergodicity lesson:')
print('  AR(1): all 30 paths converge to the same mean — time averaging works!')
print('  Random walk: each path goes its own way — the sample mean is meaningless.')
print('  Practical implication: always test for stationarity BEFORE computing any statistics.')

---
## Module 2 — Summary

| Concept | Key result | Why it matters |
|---|---|---|
| Lag operator $L$ | $Ly_t = y_{t-1}$, $\alpha(L)y_t = \varepsilon_t$ | Compact notation for all ARMA models |
| Weak stationarity | Constant $\mu$, $\gamma(0)$, $\gamma(h)$ | Pre-requisite for all ARMA machinery |
| AR(1) stationarity | $|\alpha_1| < 1$ ↔ roots outside unit circle | Coefficient constraint for valid models |
| AR(2) stationarity triangle | $|\alpha_2|<1$, $\alpha_1+\alpha_2<1$, $\alpha_2-\alpha_1<1$ | Feasible parameter space |
| Causality | AR roots outside unit circle | $y_t$ depends on past shocks only |
| Invertibility | MA roots outside unit circle | Unique, estimable MA representation |
| AR(1) ACVF | $\gamma(h) = \alpha^h \sigma^2 / (1-\alpha^2)$ | Foundation of ACF derivation |
| ACF patterns | AR: tails off; MA: cuts off at $q$ | Visual identification tool |
| PACF patterns | AR: cuts off at $p$; MA: tails off | Key for Box-Jenkins (Module 4) |
| Wold theorem | Every stationary process = MA($\infty$) | Justifies ARMA approximation |
| Ergodicity | Time avg → population avg | Why backtest statistics are valid |

---

## ➜ Next: Module 3 — AR, MA, ARMA in Depth
**AR(p) · MA(q) · ARMA(p,q) · Theoretical ACF and PACF for each family · ARMAX · ARIMA**

Module 3 goes deeper into each process family: computing theoretical ACF and PACF analytically, simulating with different parameters, and fitting AR/MA/ARMA to German bond yields, DAX sector returns, and Bundesbank money supply data.

---
*Notebook by Claude · Based on Wohlrabe UTSA 2024 · For educational use*